# ML-02 — Research Question and Provisional Lane

## 1. My lane (or freestyle) and why

In [15]:
from IPython.display import Markdown

# Read the content of skills/README.md from the cloned repository
with open('flyrank-ml-assignment/skills/README.md', 'r') as f:
    readme_content = f.read()

# Display the content as Markdown
display(Markdown(readme_content))

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write + verify the data contract (ML-04) | `writing-data-contracts/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Query the big warehouse without downloading it (ML-04/05, capstone) | `querying-big-datasets/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| EDA + signal tests with verdicts (ML-06) | `auditing-signals/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Build the rule baseline + ranked queue (ML-07) | `building-baselines/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Train and compare the model (ML-08) | `training-honest-models/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Hunt leakage; validate honestly (ML-05, ML-09) | `hunting-leakage-and-validating/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write claims that hold (ML-09, ML-10, the paper) | `writing-honest-claims/SKILL.md` | — |
| Write the research paper (ML-11, W7) | `writing-research-papers/SKILL.md` | — |
| Deploy the paper as a page (ML-11) | `deploying-static-pages/SKILL.md` | — |
| Understand FlyRank + the problem (background) | `flyrank/flyrank-context/SKILL.md` | — |

## Reuse this on your next project

Every skill outside `flyrank/` is **general** — take this whole folder to any future project.
Delete `skills/flyrank/` and the flyrank column above, and everything else still works.


In [16]:
from IPython.display import Markdown

# Read the content of framing-ml-problems/SKILL.md
print("--- Content of framing-ml-problems/SKILL.md ---")
with open('flyrank-ml-assignment/skills/framing-ml-problems/SKILL.md', 'r') as f:
    skill_content_framing = f.read()
display(Markdown(skill_content_framing))

--- Content of framing-ml-problems/SKILL.md ---


---
name: framing-ml-problems
description: Frames a data/ML problem before any modeling — the decision, the action, the cost of a wrong call, task type, target, and success metric. Use when picking a project direction, writing a research question, or mapping a problem onto an ML task type.
---

# Framing ML problems

A model is never the goal. A better DECISION is the goal. Frame first, model later.

## The four questions (answer all four, in writing)

1. **What decision does this improve?** Not "predict X" — decisions. "Which page should an
   editor fix first?" is a decision. "Predict decline" is not, yet.
2. **Who acts on the output, and what do they do?** Name the person and the action. If nobody
   would act differently, the work has no customer.
3. **What does a wrong answer cost?** Wasted editor hours? A missed decline? The cost of errors
   decides how careful the method must be, and which errors matter more.
4. **Why does data or ML help at all?** Sometimes a plain rule (an if-statement) is the right
   answer. Sometimes a dashboard is. ML earns its place only when the pattern is real but too
   messy to write by hand — many signals, tangled, shifting over time.

## Map it to a task type

| If your question sounds like… | Task type | Target | Typical metric |
|---|---|---|---|
| "Which ones first?" | Ranking / scoring | a priority score | precision@K |
| "Will this one decline / recover?" | Classification | a yes/no label from an OBSERVED outcome | ROC-AUC, precision/recall vs base rate |
| "What kinds of items exist?" | Clustering | none (unsupervised) | silhouette + human sense-check |
| "Which signals travel together?" | Signal analysis | none | effect sizes, grouped comparisons |

Two rules that save whole projects:

- **The target must be observed, not defined.** A label that comes from someone's rule means your
  model learns the rule, not the world. Prefer outcomes measured in a later time window.
- **Name the metric before training.** "Good" defined after the fact always looks good.

## Write the one-paragraph frame

> For [who], deciding [what], we will build [output type] from [data], predicting/scoring
> [target] measured by [metric]. A wrong call costs [cost]. A plain rule isn't enough because
> [why]. We will claim only [observed / directional / decision-support] results.

If you can fill that paragraph honestly, you are framed. If a blank stays blank, that blank IS
your next task.

## How to verify

- Show your frame to someone (or your assistant) and ask: "what decision does this improve?"
  If their answer differs from yours, the frame isn't done.
- Check the target: is it observed in the data, or defined by a rule? If defined — reframe.
- Check the metric exists in your data. Can you compute it today, on a baseline? Do it.


In [17]:
from IPython.display import Markdown

# Read the content of flyrank/flyrank-data/SKILL.md
print("\n--- Content of flyrank/flyrank-data/SKILL.md ---")
with open('flyrank-ml-assignment/skills/flyrank/flyrank-data/SKILL.md', 'r') as f:
    skill_content_flyrank_data = f.read()
display(Markdown(skill_content_flyrank_data))


--- Content of flyrank/flyrank-data/SKILL.md ---


---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one ships in this repo; the big one is hosted and gated.

## 1. Starter dataset (in this repo)

`data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized
content item, 32 clients, trailing-90-day metrics. Full column reference: `docs/data-dictionary.md`
(keep it open). The gotchas that cause 90% of mistakes:

- **Rate columns are ×100 percentages**: `ctr = 0.76` means 0.76%, not 76%. Applies to ctr,
  engagement_rate, scroll_rate, ai_traffic_pct, trend_pct.
- **`avg_position = 0` means "no data"**, not rank zero (1,205 rows).
- **`scroll_rate` and `ai_traffic_pct` can exceed 100** — numerator and denominator come from
  different measurement systems. Not a bug; read the dictionary.
- **The label trap:** `is_declining_label` is derived from `trend_direction`, which is computed
  from `trend_pct`. Therefore `trend_direction` and `trend_pct` are NEVER features.
- **Missingness follows content_type** (one type has ~100% missing keyword data; another ~28%
  missing word_count) — a blind fillna(0) injects a category signal. Add has_-flags instead.
- **IDs (`content_id`, `client_id`) are pseudonyms:** grouping/joining/splitting only, never
  features. Use client_id for grouped train/test splits.

## 2. Warehouse release (Hugging Face, gated — instant approval)

`hf://datasets/FlyRank/internship-warehouse` — build v20260703:

| Table | Rows | Grain |
|---|---|---|
| `dim_clients` | 104 | one per pseudonymized client |
| `dim_content` | 519,606 | one per content item |
| `fact_content_daily_performance` | 78,835,655 | report_date × client × content (partitioned by month) |
| `fact_content_daily_performance_sample` | ~11.7M | same grain — the FINAL MONTH (June 2026) only; fine for query mechanics, NEVER for label logic |
| `fact_content_query_90d` | 2,414,248 | client × content × query hash, fixed 90-day window |

Dates: 2025-01-27 → 2026-06-30 (~17 months). Position column is `gsc_avg_position`.

**Panel warnings (they are real):**
- History depth differs wildly per client — check `dim_clients.gsc_data_start` before defining
  any time window; prefer per-client windows over one global calendar window.
- Rows before a client's `ga4_data_start` have GA4 columns zero-FILLED with
  `ga4_data_available = FALSE` — filter on the flag; zeros there are not "no engagement".
- A third of clients have little or no usable search/analytics history — expect to filter.
- The query table's per-content context columns repeat on every row: `ANY_VALUE()` them, never
  `SUM()`. Its 90-day window overlaps the snapshot's final months — if your label lives in the
  last 30 days, only `*_prev30`-style columns are safe features (window alignment first!).

**Access + iteration rules:** request gate access once (instant), READ token via Colab
Secrets (`HF_TOKEN`) or env var — never pasted in a cell (public repo!). Iterate on a
**mid-panel month partition** (e.g. `month=2026-03`), not the `_sample`: the `_sample` is the
panel's LAST month, i.e. the natural outcome window of any past→future label — develop label
logic there and you are developing inside your own test window. Treat the final month as a
sealed test month. Run the full 79M scan ONCE when the query is final and cache the result to
`work/outputs/`. Repeated full scans hit HTTP 429 rate limits.

## How to verify

- Start any session with `COUNT(*)` + `MIN/MAX(report_date)` on your table and match the
  numbers above.
- Grain-probe before aggregating: `GROUP BY <grain cols> HAVING COUNT(*) > 1 LIMIT 5` → empty.
- If a number looks absurd (a 400% rate, a giant ratio from 12 rows), re-read the gotchas
  before celebrating.


In [18]:
import os

# Get the current working directory
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")

# List contents of the current directory to find 'skills' folder
print("Contents of current directory:")
for item in os.listdir(current_dir):
    print(item)

# Also list contents of the 'work' directory if it exists, as the path hint suggests 'work/notebooks/'
work_path = os.path.join(current_dir, 'work')
if os.path.exists(work_path) and os.path.isdir(work_path):
    print(f"\nContents of 'work' directory ({work_path}):")
    for item in os.listdir(work_path):
        print(item)

    # Check for 'skills' inside 'work'
    skills_in_work_path = os.path.join(work_path, 'skills')
    if os.path.exists(skills_in_work_path) and os.path.isdir(skills_in_work_path):
        print(f"\nContents of 'work/skills' directory ({skills_in_work_path}):")
        for item in os.listdir(skills_in_work_path):
            print(item)

Current working directory: /content
Contents of current directory:
.config
flyrank-ml-assignment
sample_data


In [19]:
!git clone https://github.com/ayush0121n/flyrank-ml-assignment.git

import os
# List contents of the cloned directory to confirm 'skills' folder
cloned_repo_path = 'flyrank-ml-assignment'
if os.path.exists(cloned_repo_path) and os.path.isdir(cloned_repo_path):
    print(f"\nContents of '{cloned_repo_path}' directory:")
    for item in os.listdir(cloned_repo_path):
        print(item)

    # Check for 'skills' inside the cloned repo
    skills_in_cloned_repo_path = os.path.join(cloned_repo_path, 'skills')
    if os.path.exists(skills_in_cloned_repo_path) and os.path.isdir(skills_in_cloned_repo_path):
        print(f"\nContents of '{cloned_repo_path}/skills' directory:")
        for item in os.listdir(skills_in_cloned_repo_path):
            print(item)

fatal: destination path 'flyrank-ml-assignment' already exists and is not an empty directory.

Contents of 'flyrank-ml-assignment' directory:
LICENSE
notebooks
skills
.github
CLAUDE.md
data
.git
submission
AGENTS.md
docs
requirements.txt
outputs
README.md
work
DATA_USE.md
GUIDE.md
.gitignore
SETUP.md
scripts

Contents of 'flyrank-ml-assignment/skills' directory:
writing-data-contracts
querying-big-datasets
training-honest-models
writing-research-papers
flyrank
deploying-static-pages
framing-ml-problems
README.md
writing-honest-claims
directing-your-ai-assistant
hunting-leakage-and-validating
building-baselines
auditing-signals


**Chosen Lane:** Lane 1 — Content Decay and Refresh Optimization.

**Why this one:** In the starter dataset, a significant portion of content items exhibit operational performance drops over time, making manual tracking impossible for large-scale editorial teams. By focusing on predictive classification of content decay, we can catch dropping performance early before organic search real estate is lost completely.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call
* **What decision does this improve?** It improves the editorial team's decision on which specific URL or content asset to prioritize for an optimization refresh next week.
* **Who acts on the output, and what do they do?** The Content Editor / SEO Manager acts on it. They will open the top-ranked "at-risk" articles and update the out-of-date sections, facts, or keywords.
* **What does a wrong answer cost?** A false positive wastes hours of manual editorial time refreshing an article that didn't need it. A false negative (missing a declining page) leads to a long-term drop in organic traffic, lowering ad impressions or conversion revenue.
* **Why does data or ML help at all?** Traffic patterns shift dynamically across thousands of URLs across 32 different clients. A simple rule baseline cannot scale across different industries or capture the messy combinations of impression drops, positions, and scroll-rate reductions simultaneously.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)
*The starter dataset has been analyzed below to back up this choice with real baseline numbers:*

In [25]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv('flyrank-ml-assignment/data/raw/content_refresh_anonymized.csv')

# Create 'is_declining_label' based on 'trend_direction'
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Calculate baseline numbers
total_rows = len(df)
declining_count = df['is_declining_label'].sum()
pct_declining = (declining_count / total_rows) * 100

print(f"Total Rows: {total_rows}")
print(f"Declining Items: {declining_count} ({pct_declining:.2f}%)")

Total Rows: 30000
Declining Items: 16262 (54.21%)


## 4. Careful words: what I can and can't claim
We will claim only directional decision-support results. This model scores priority and alerts editors to observed correlation patterns in historical performance windows. It does not provide causal proof of why Google ranks a page lower, nor does it guarantee that updating an article will magically restore 100% of its traffic.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Self-check
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.